<a href="https://colab.research.google.com/github/XAMANEKR/compaplica_mita/blob/main/Copia_de_Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar los modelos que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [1]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga de modelos.

In [2]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [3]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda

import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.0


### **DESCARGAR LOS MODELOS**

Vamos a trabajar con dos tamaños de Llama: **Llama 3.2 de 3B** (ligero) y **Llama 3.1 de 8B** (más grande). Guardamos los nombres en variables para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [4]:
# Descargar los dos modelos Llama (la primera vez tarda unos minutos)

MODELO_LIGERO = "llama3.2:3b"
MODELO_GRANDE = "llama3.1:8b"

!ollama pull {MODELO_LIGERO}
!ollama pull {MODELO_GRANDE}
!ollama list



NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    57 seconds ago            


### **CLIENTE DE PYTHON**

In [5]:
# Instalar el cliente de Ollama y conectarlo al servidor local

!pip install ollama -q

import ollama
import time

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **DE PALABRAS A TOKENS**

### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**.

Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [6]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)

print(prompt)
print(response.message.content)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?
¡Excelente pregunta!

La RAM (Random Access Memory) y el almacenamiento en una computadora son dos conceptos diferentes, aunque están relacionados entre sí.

**La RAM (RAM)**

La RAM es un tipo de memoria de acceso aleatorio, que significa que la computadora puede acceder rápidamente a cualquier dato almacenado en ella. La RAM es utilizada para almacenar datos y programas que la computadora está utilizando en el momento presente. Cuando un programa o aplicación está ejecutándose, se carga en la RAM, lo que permite a la computadora acceder a él rápidamente.

La RAM es un recurso limitado y puede ser considerada como una "celda de trabajo" para la computadora. Cuando la computadora necesita realizar una tarea, se le asigna una cantidad de RAM para trabajar con los datos necesarios. Si la computadora necesita acceder a más datos, puede pedirle más RAM a la placa base, pero esto puede ser limitado por la cantidad d

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de Ollama trae el conteo de tokens de entrada (`prompt_eval_count`) y de salida (`eval_count`). Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [7]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

tokens_prompt = response.prompt_eval_count
tokens_respuesta = response.eval_count

print("Tokens del prompt:", tokens_prompt)
print("Tokens de la respuesta:", tokens_respuesta)
print("Tokens totales:", tokens_prompt + tokens_respuesta)
# Con Ollama no se factura: los tokens miden cuánto trabajo hizo el modelo y cuánto de su contexto usaste

Tokens del prompt: 43
Tokens de la respuesta: 554
Tokens totales: 597


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [8]:
# Medir el tiempo de respuesta de Llama para el mismo prompt

inicio = time.time()
response_tiempo = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 6.85 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama que cabe en la GPU gratuita de Colab (8B) y compara tiempo de respuesta y calidad contra el modelo ligero (3B).

Ojo: la primera vez que se usa un modelo, Ollama tiene que **cargarlo en la memoria de la GPU**. Ese tiempo de carga viene en `load_duration` y conviene separarlo para que la comparación sea justa.

In [9]:

# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad

inicio = time.time()
response_grande = client.chat(
    model=MODELO_GRANDE,
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio
carga_grande = response_grande.load_duration / 1e9  # nanosegundos -> segundos

tokens_ligero = response.prompt_eval_count + response.eval_count
tokens_grande = response_grande.prompt_eval_count + response_grande.eval_count

print(f"Modelo ligero ({MODELO_LIGERO}): {duracion:.2f} s — {tokens_ligero} tokens")
print(f"Modelo grande ({MODELO_GRANDE}): {duracion_grande:.2f} s — {tokens_grande} tokens (de los cuales {carga_grande:.2f} s fueron carga del modelo)")
print("\nRespuesta del modelo grande:\n", response_grande.message.content)

Modelo ligero (llama3.2:3b): 6.85 s — 597 tokens
Modelo grande (llama3.1:8b): 21.96 s — 156 tokens (de los cuales 18.18 s fueron carga del modelo)

Respuesta del modelo grande:
 La diferencia entre la RAM (Memoria de Acceso Aleatorio) y el almacenamiento es la siguiente:

* La RAM es una memoria que utiliza la computadora mientras la esté funcionando, la información que se almacena en la RAM se perde cuando la computadora se apaga, la RAM se utiliza para almacenar la información que la computadora necesita para ejecutar sus programas y tareas de manera rápida.
* El almacenamiento es un dispositivo que utiliza la computadora para guardar y recuperar datos, ya sea en una unidad de disco duro, un disco externo o una tarjeta de memoria.


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:** Prepara el entorno de configuración del entorno con **Ollama** activando la **T4 GPU** e inicializa el cliente.

In [10]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)


In [11]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda


In [12]:
# Descargar el modelo ligero de Llama e inicializar el cliente


**2. Consulta la primera pregunta:**

* Construye una lista llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).
*  Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [13]:
# Definir la lista de preguntas


In [14]:
# Consultar la pregunta 1 y guardar el resultado en resultado_1


**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [15]:
# Consultar la pregunta 2 y guardar el resultado en resultado_2


In [16]:
# Consultar la pregunta 3 y guardar el resultado en resultado_3


**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [17]:
# Definir la lista resultados y agregar los tres diccionarios


**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [18]:
# Mostrar la tabla final de resultados
